# Milestone 6: Evaluation (Updated for Report)

Computes Precision@K, Recall@K, and NDCG@K for all four models at K = 1, 5, 10, 15, 20.
Prints all values and saves all figures needed for the final report.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/MLG Project/Project'
OUT_DIR = f'{BASE}/data/processed'
FIG_DIR = f'{BASE}/results/figures'

import os
os.makedirs(FIG_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## Load all scored candidate files

In [3]:
# Baseline scores (popularity + MF) from M4
val_scored  = pd.read_csv(f'{OUT_DIR}/val_candidates_scored.csv')
test_scored = pd.read_csv(f'{OUT_DIR}/test_candidates_scored.csv')

# GraphSAGE scores from M5
val_gs  = pd.read_csv(f'{OUT_DIR}/val_candidates_graphsage_scored.csv')[['userId','movieId','graphsage_score']]
test_gs = pd.read_csv(f'{OUT_DIR}/test_candidates_graphsage_scored.csv')[['userId','movieId','graphsage_score']]

# LightGCN embeddings + mappings from M5
user_emb  = np.load(f'{OUT_DIR}/user_emb.npy')
movie_emb = np.load(f'{OUT_DIR}/movie_emb.npy')

user2idx  = pd.read_csv(f'{OUT_DIR}/user2idx.csv').set_index('userId')['idx'].to_dict()
movie2idx = pd.read_csv(f'{OUT_DIR}/movie2idx.csv').set_index('movieId')['idx'].to_dict()

NUM_USERS = user_emb.shape[0]

print(f'Val candidates:    {len(val_scored)}')
print(f'Test candidates:   {len(test_scored)}')
print(f'User emb shape:    {user_emb.shape}')
print(f'Movie emb shape:   {movie_emb.shape}')

Val candidates:    19278
Test candidates:   23256
User emb shape:    (609, 128)
Movie emb shape:   (6298, 128)


## Score LightGCN candidates using saved embeddings

In [4]:
def score_lightgcn(df):
    scores = []
    for row in df.itertuples():
        uid, mid = row.userId, row.movieId
        if uid in user2idx and mid in movie2idx:
            u_vec = user_emb[user2idx[uid]]
            m_vec = movie_emb[movie2idx[mid] - NUM_USERS]
            scores.append(float(u_vec @ m_vec))
        else:
            scores.append(np.nan)
    return scores

val_scored['lightgcn_score']  = score_lightgcn(val_scored)
test_scored['lightgcn_score'] = score_lightgcn(test_scored)

print('LightGCN scoring complete.')
print(f'NaN in val:  {val_scored["lightgcn_score"].isna().sum()}')
print(f'NaN in test: {test_scored["lightgcn_score"].isna().sum()}')

LightGCN scoring complete.
NaN in val:  0
NaN in test: 0


## Merge all scores

In [5]:
val_full  = val_scored.merge(val_gs,  on=['userId','movieId'], how='left')
test_full = test_scored.merge(test_gs, on=['userId','movieId'], how='left')

print('Columns:', test_full.columns.tolist())

Columns: ['userId', 'movieId', 'label', 'split', 'popularity_score', 'mf_score', 'lightgcn_score', 'graphsage_score']


## Evaluation functions

In [6]:
def precision_at_k(group, score_col, k):
    top = group.nlargest(k, score_col)
    return top['label'].sum() / k

def recall_at_k(group, score_col, k):
    top = group.nlargest(k, score_col)
    total_pos = group['label'].sum()
    if total_pos == 0:
        return np.nan
    return top['label'].sum() / total_pos

def ndcg_at_k(group, score_col, k):
    top = group.nlargest(k, score_col).reset_index(drop=True)
    total_pos = group['label'].sum()
    if total_pos == 0:
        return np.nan
    dcg  = sum(top['label'].iloc[i] / np.log2(i + 2) for i in range(len(top)))
    ideal_hits = min(int(total_pos), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(df, score_col, k=10):
    precisions, recalls, ndcgs = [], [], []
    for _, group in df.groupby('userId'):
        if group['label'].sum() == 0:
            continue
        precisions.append(precision_at_k(group, score_col, k))
        recalls.append(recall_at_k(group, score_col, k))
        ndcgs.append(ndcg_at_k(group, score_col, k))
    return {
        f'Precision@{k}': round(float(np.nanmean(precisions)), 4),
        f'Recall@{k}':    round(float(np.nanmean(recalls)),    4),
        f'NDCG@{k}':      round(float(np.nanmean(ndcgs)),      4),
    }

print('Evaluation functions ready.')

Evaluation functions ready.


## Results at K = 10 (main results table)

In [7]:
models = {
    'Popularity':   'popularity_score',
    'Matrix Fact.': 'mf_score',
    'GraphSAGE':    'graphsage_score',
    'LightGCN':     'lightgcn_score',
}

report_table = pd.DataFrame([
    {'Model': name, **evaluate(test_full, col, 10)}
    for name, col in models.items()
]).set_index('Model')

print('── Test Set Results @ K=10 ──')
print(report_table.to_string())

── Test Set Results @ K=10 ──
              Precision@10  Recall@10  NDCG@10
Model                                         
Popularity          0.2417     0.1850   0.2531
Matrix Fact.        0.0417     0.0557   0.1160
GraphSAGE           0.1917     0.1366   0.2204
LightGCN            0.3083     0.2445   0.3090


## Results across K = 1, 5, 10, 15, 20 (printed for report)

In [8]:
K_values = [1, 5, 10, 15, 20]

# Collect all results
all_results = {name: {} for name in models}
for k in K_values:
    for name, col in models.items():
        m = evaluate(test_full, col, k)
        all_results[name][k] = m

# Print full table for every metric at every K — copy these into the report
for metric in ['Precision', 'Recall', 'NDCG']:
    print(f'\n── {metric}@K ──')
    header = f'{"Model":<20}' + ''.join(f'  K={k:<6}' for k in K_values)
    print(header)
    print('-' * len(header))
    for name in models:
        row = f'{name:<20}' + ''.join(
            f'  {all_results[name][k][f"{metric}@{k}"]:<8.4f}'
            for k in K_values
        )
        print(row)


── Precision@K ──
Model                 K=1       K=5       K=10      K=15      K=20    
----------------------------------------------------------------------
Popularity            0.2083    0.2500    0.2417    0.1944    0.1812  
Matrix Fact.          0.4167    0.0833    0.0417    0.0278    0.0208  
GraphSAGE             0.2500    0.2333    0.1917    0.1944    0.1750  
LightGCN              0.2083    0.3000    0.3083    0.2750    0.2458  

── Recall@K ──
Model                 K=1       K=5       K=10      K=15      K=20    
----------------------------------------------------------------------
Popularity            0.0156    0.0999    0.1850    0.2334    0.2874  
Matrix Fact.          0.0557    0.0557    0.0557    0.0557    0.0557  
GraphSAGE             0.0146    0.0895    0.1366    0.2167    0.2845  
LightGCN              0.0146    0.1288    0.2445    0.3507    0.4086  

── NDCG@K ──
Model                 K=1       K=5       K=10      K=15      K=20    
----------------------------

## Figure 1 — Bar chart at K=10
Saved as `fig_barcharts.png`

In [9]:
COLORS = ['#5DCAA5', '#AFA9EC', '#F0997B', '#7F77DD']  # pop, mf, sage, lgcn
MODEL_LABELS = ['Popularity', 'Matrix Fact.', 'GraphSAGE', 'LightGCN']

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e8e8e8',
    'grid.linewidth': 0.6,
})

metrics_k10 = {
    'Precision@10': [all_results[n][10][f'Precision@10'] for n in models],
    'Recall@10':    [all_results[n][10][f'Recall@10']    for n in models],
    'NDCG@10':      [all_results[n][10][f'NDCG@10']      for n in models],
}

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
x = np.arange(len(MODEL_LABELS))

for ax, (metric, vals) in zip(axes, metrics_k10.items()):
    bars = ax.bar(x, vals, color=COLORS, width=0.55, zorder=3,
                  edgecolor='white', linewidth=0.5)
    ax.set_title(metric, fontsize=10, fontweight='bold', pad=6)
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, rotation=20, ha='right', fontsize=8.5)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.2f}'))
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=7.5, color='#444')

plt.tight_layout(pad=1.2)
path_bar = f'{FIG_DIR}/fig_barcharts.png'
plt.savefig(path_bar, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: {path_bar}')

Saved: /content/drive/MyDrive/MLG Project/Project/results/figures/fig_barcharts.png


## Figure 2 — Line charts across K values
Saved as `fig_linecharts.png`

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
MARKERS = ['o', 's', '^', 'D']

for ax, metric in zip(axes, ['Precision', 'Recall', 'NDCG']):
    for (name, col), color, marker in zip(models.items(), COLORS, MARKERS):
        vals = [all_results[name][k][f'{metric}@{k}'] for k in K_values]
        ax.plot(K_values, vals, color=color, marker=marker,
                markersize=5, linewidth=1.8, label=name)
    ax.set_title(f'{metric}@K', fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('K', fontsize=9)
    ax.set_xticks(K_values)
    ax.set_ylim(bottom=0)
    ax.grid(True, color='#e8e8e8', linewidth=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles = [
    plt.Line2D([0], [0], color=c, marker=m, markersize=6, linewidth=1.8, label=l)
    for c, m, l in zip(COLORS, MARKERS, MODEL_LABELS)
]
fig.legend(handles=handles, loc='lower center', ncol=4,
           fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout(pad=1.2)
path_line = f'{FIG_DIR}/fig_linecharts.png'
plt.savefig(path_line, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
print(f'Saved: {path_line}')

Saved: /content/drive/MyDrive/MLG Project/Project/results/figures/fig_linecharts.png


## Save all results to CSV

In [11]:
# Build a tidy CSV with all models x all K values
rows = []
for name in models:
    for k in K_values:
        r = all_results[name][k]
        rows.append({
            'Model': name, 'K': k,
            'Precision': r[f'Precision@{k}'],
            'Recall':    r[f'Recall@{k}'],
            'NDCG':      r[f'NDCG@{k}'],
        })

results_df = pd.DataFrame(rows)
csv_path = f'{OUT_DIR}/final_results_all_k.csv'
results_df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
print('\nFull results table:')
print(results_df.to_string(index=False))
print('\nDone.')

Saved: /content/drive/MyDrive/MLG Project/Project/data/processed/final_results_all_k.csv

Full results table:
       Model  K  Precision  Recall   NDCG
  Popularity  1     0.2083  0.0156 0.2083
  Popularity  5     0.2500  0.0999 0.2447
  Popularity 10     0.2417  0.1850 0.2531
  Popularity 15     0.1944  0.2334 0.2539
  Popularity 20     0.1812  0.2874 0.2695
Matrix Fact.  1     0.4167  0.0557 0.4167
Matrix Fact.  5     0.0833  0.0557 0.1489
Matrix Fact. 10     0.0417  0.0557 0.1160
Matrix Fact. 15     0.0278  0.0557 0.1098
Matrix Fact. 20     0.0208  0.0557 0.1069
   GraphSAGE  1     0.2500  0.0146 0.2500
   GraphSAGE  5     0.2333  0.0895 0.2450
   GraphSAGE 10     0.1917  0.1366 0.2204
   GraphSAGE 15     0.1944  0.2167 0.2452
   GraphSAGE 20     0.1750  0.2845 0.2587
    LightGCN  1     0.2083  0.0146 0.2083
    LightGCN  5     0.3000  0.1288 0.2820
    LightGCN 10     0.3083  0.2445 0.3090
    LightGCN 15     0.2750  0.3507 0.3373
    LightGCN 20     0.2458  0.4086 0.3529

Done.
